In [ ]:
# Locate the repository when running from any notebook subfolder.
from pathlib import Path
import os
_notebook_start = Path.cwd().resolve()
_notebook_root = next(
    (p for p in [_notebook_start, *_notebook_start.parents]
     if (p / "docs" / "predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if _notebook_root is None:
    raise FileNotFoundError("Cannot locate the project repository.")
os.chdir(_notebook_root)


In [1]:
# Purpose: Load the research table, cleaned items, and cleaned products.

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()

if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

research = pd.read_csv(
    DATA_DIR / "research_data.csv",
    dtype={"order_id": "string"}
)

items = pd.read_csv(
    DATA_DIR / "items_clean.csv",
    dtype={"order_id": "string", "product_id": "string"}
)

products = pd.read_csv(
    DATA_DIR / "products_clean.csv",
    dtype={"product_id": "string",
           "product_category_name": "string"}
)

print("Research:", research.shape)
print("Items:", items.shape)
print("Products:", products.shape)

Research: (99441, 34)
Items: (112650, 7)
Products: (32951, 14)


In [2]:
# Purpose: Check keys and verify existing order totals.

assert research["order_id"].notna().all()
assert research["order_id"].is_unique

assert products["product_id"].notna().all()
assert products["product_id"].is_unique

assert not items.duplicated(
    ["order_id", "order_item_id"]
).any()

for column in ["price", "freight_value"]:
    items[column] = pd.to_numeric(items[column], errors="raise")

totals = items.groupby("order_id").agg(
    items_value=("price", lambda x: x.sum(min_count=1)),
    freight_value=("freight_value", lambda x: x.sum(min_count=1)),
    item_count=("order_item_id", "size")
)

comparison = research.merge(
    totals,
    on="order_id",
    how="left",
    suffixes=("", "_check"),
    validate="one_to_one"
)

for column in ["items_value", "freight_value", "item_count"]:
    original = comparison[column]
    recalculated = comparison[f"{column}_check"]

    assert np.allclose(
        original,
        recalculated,
        equal_nan=True
    ), f"Order totals do not match: {column}"

print("Keys and order totals passed.")
print("Orders without item records:", comparison["item_count_check"].isna().sum())

Keys and order totals passed.
Orders without item records: 775


In [3]:
# Purpose: Create price features while preserving missing values.

enriched = research.copy()

valid_value = enriched["items_value"].where(
    enriched["items_value"] > 0
)

valid_count = enriched["item_count"].where(
    enriched["item_count"] > 0
)

enriched["order_total_value"] = (
    enriched["items_value"] + enriched["freight_value"]
)

enriched["freight_to_items_ratio"] = (
    enriched["freight_value"] / valid_value
)

enriched["mean_item_value"] = (
    enriched["items_value"] / valid_count
)

enriched["freight_per_item"] = (
    enriched["freight_value"] / valid_count
)

price_features = [
    "order_total_value",
    "freight_to_items_ratio",
    "mean_item_value",
    "freight_per_item"
]

display(enriched[["order_id"] + price_features].head())

,order_id,order_total_value,freight_to_items_ratio,mean_item_value,freight_per_item
0,e481f51cbdc54678b7cc49136f2d6af7,38.71,0.290764,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,141.46,0.191744,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,179.12,0.120200,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,72.20,0.604444,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,28.62,0.438191,19.90,8.72


In [4]:
# Purpose: Attach product information to each item.

product_columns = [
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

item_products = items.merge(
    products[product_columns],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert len(item_products) == len(items)

unmatched = item_products["_merge"].eq("left_only").sum()

print("Items without a matching product:", unmatched)
assert unmatched == 0

item_products = item_products.drop(columns="_merge")

Items without a matching product: 0


In [5]:
# Purpose: Prepare valid item weight, volume, and category values.

measurements = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in measurements:
    values = pd.to_numeric(
        item_products[column],
        errors="raise"
    )

    item_products[column] = values.where(
        values.gt(0) & np.isfinite(values)
    )

item_products["item_weight_g"] = item_products["product_weight_g"]

item_products["item_volume_cm3"] = (
    item_products["product_length_cm"]
    * item_products["product_height_cm"]
    * item_products["product_width_cm"]
)

item_products["item_volume_cm3"] = (
    item_products["item_volume_cm3"].where(
        np.isfinite(item_products["item_volume_cm3"])
    )
)

category = item_products["product_category_name"].str.strip()

item_products["product_category_name"] = category.replace(
    "", pd.NA
)

display(
    item_products[
        ["item_weight_g", "item_volume_cm3", "product_category_name"]
    ].isna().sum().to_frame("missing_rows")
)

,missing_rows
item_weight_g,26
item_volume_cm3,18
product_category_name,1603


In [6]:
# Purpose: Aggregate product information to one row per order.

product_summary = item_products.groupby("order_id").agg(
    product_item_count=("product_id", "size"),

    known_weight_items=("item_weight_g", "count"),
    known_volume_items=("item_volume_cm3", "count"),
    known_category_items=("product_category_name", "count"),

    total_product_weight_g=(
        "item_weight_g", lambda x: x.sum(min_count=1)
    ),
    total_product_volume_cm3=(
        "item_volume_cm3", lambda x: x.sum(min_count=1)
    ),
    product_category_count=("product_category_name", "nunique")
)

for name in ["weight", "volume", "category"]:
    known = product_summary[f"known_{name}_items"]
    total = product_summary["product_item_count"]

    product_summary[f"{name}_coverage_ratio"] = known / total

complete_rules = {
    "total_product_weight_g": "weight_coverage_ratio",
    "total_product_volume_cm3": "volume_coverage_ratio",
    "product_category_count": "category_coverage_ratio"
}

for feature, coverage in complete_rules.items():
    product_summary[feature] = product_summary[feature].where(
        product_summary[coverage].eq(1)
    )

product_features = list(complete_rules) + [
    "weight_coverage_ratio",
    "volume_coverage_ratio",
    "category_coverage_ratio"
]

enriched = enriched.merge(
    product_summary[product_features],
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [7]:
# Purpose: Validate the enriched table and save a separate output.

new_features = price_features + product_features

assert len(enriched) == len(research)
assert enriched["order_id"].is_unique

pd.testing.assert_frame_equal(
    enriched[research.columns],
    research
)

for column in new_features:
    values = enriched[column].dropna()

    assert np.isfinite(values).all(), f"Infinite values: {column}"
    assert values.ge(0).all(), f"Negative values: {column}"

for column in [
    "weight_coverage_ratio",
    "volume_coverage_ratio",
    "category_coverage_ratio"
]:
    assert enriched[column].dropna().between(0, 1).all()

missing_summary = pd.DataFrame({
    "missing_rows": enriched[new_features].isna().sum(),
    "missing_percent": (
        enriched[new_features].isna().mean() * 100
    ).round(3)
})

display(missing_summary)

output_path = DATA_DIR / "research_data_price_product.csv"
enriched.to_csv(output_path, index=False)

print("All checks passed.")
print("Saved:", output_path)
print("Dataset shape:", enriched.shape)

,missing_rows,missing_percent
order_total_value,775,0.779
freight_to_items_ratio,775,0.779
mean_item_value,775,0.779
freight_per_item,775,0.779
total_product_weight_g,797,0.801
total_product_volume_cm3,791,0.795
product_category_count,2226,2.239
weight_coverage_ratio,775,0.779
volume_coverage_ratio,775,0.779
category_coverage_ratio,775,0.779


All checks passed.
Saved: c:\Users\LENOVO\OneDrive\Desktop\PTTK\Group5_STAT3013.R11\data\processed\research_data_price_product.csv
Dataset shape: (99441, 44)
